# Casualty Generator
Reads the military force structure, then generates a casualty dataset from waves.

In [206]:
import pandas as pd
from pathlib import Path

# ── Load force structure ───────────────────────────────────────────────────────
FORCE_STRUCTURE_FILE = (
    "data/single_run/input/casualty_generation_settings/"
    "military_force_structure/force_structure_400.xlsx"
)

fs_df = pd.read_excel(FORCE_STRUCTURE_FILE)
fs_df["spawning_points"] = fs_df["spawning_points"].apply(
    lambda x: [s.strip() for s in str(x).split(",") if s.strip()]
)

# ── Summary ────────────────────────────────────────────────────────────────────
involved   = fs_df[fs_df["involved"].astype(bool)]
n_div      = fs_df["division"].nunique()
n_bde      = fs_df["brigade"].nunique()
n_bn       = len(fs_df)
n_bn_inv   = len(involved)
total_par  = fs_df["par"].sum()
inv_par    = involved["par"].sum()

all_spawn  = sorted({sp for pts in fs_df["spawning_points"] for sp in pts})
inv_spawn  = sorted({sp for pts in involved["spawning_points"] for sp in pts})

print(f"{'─'*60}")
print(f"  MILITARY FORCE STRUCTURE  ·  {FORCE_STRUCTURE_FILE.split('/')[-1]}")
print(f"{'─'*60}")
print(f"  Divisions         : {n_div}")
print(f"  Brigades          : {n_bde}")
print(f"  Battalions        : {n_bn}  ({n_bn_inv} involved in combat)")
print(f"  Total force PAR   : {total_par:,}")
print(f"  Total PAR (combat): {inv_par:,}")
print(f"{'─'*60}")
print(f"  Spawning points — involved only ({len(inv_spawn)}):")
print(f"    {', '.join(inv_spawn) if inv_spawn else '(none)'}")
print(f"{'─'*60}")
print()
print(fs_df[["division","brigade","battalion","par","involved","spawning_points"]].to_string(index=False))

────────────────────────────────────────────────────────────
  MILITARY FORCE STRUCTURE  ·  force_structure_400.xlsx
────────────────────────────────────────────────────────────
  Divisions         : 1
  Brigades          : 3
  Battalions        : 6  (1 involved in combat)
  Total force PAR   : 18,798
  Total PAR (combat): 3,133
────────────────────────────────────────────────────────────
  Spawning points — involved only (3):
    ccp_7, ccp_8, ccp_9
────────────────────────────────────────────────────────────

division brigade battalion  par  involved          spawning_points
    DIV1    BDE1       BN1 3133     False    [ccp_1, ccp_2, ccp_3]
    DIV1    BDE1       BN2 3133     False    [ccp_4, ccp_5, ccp_6]
    DIV1    BDE2       BN3 3133      True    [ccp_7, ccp_8, ccp_9]
    DIV1    BDE2       BN4 3133     False [ccp_10, ccp_11, ccp_12]
    DIV1    BDE3       BN5 3133     False [ccp_13, ccp_14, ccp_15]
    DIV1    BDE3       BN6 3133     False [ccp_16, ccp_17, ccp_18]


In [207]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Callable, Tuple, Union
import random

import numpy as np
import pandas as pd
import salabim as sim
import yaml

In [208]:
# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_NAME   = "casualties_custom_test"          # saved as data/single_run/input/casualty_dataframes/{OUTPUT_NAME}.xlsx
SAVE_OUTPUT   = True

# ── Wave schedule ─────────────────────────────────────────────────────────────
N_DAYS            = 1
WAVES_PER_DAY     = 1
WAVE_DURATION_MIN = 30
SEED              = 11

# ── BCAS diagnosis mix ────────────────────────────────────────────────────────
bcas_mix_df = pd.DataFrame({
    "triage":                ["T1_BCAS",    "T2_BCAS",    "T3_BCAS",    "KIA",        "MIA"       ],
    "frequency":             [ 0.105,        0.140,        0.455,        0.200,        0.100       ],
    "lifetime_distribution": ["triangular",  "triangular", "triangular", "triangular", "triangular"],
    "lifetime_low":          [ 40,           1200,         10000,        1,            1           ],
    "lifetime_mode":         [ 60,           1440,         11000,        1,            1           ],
    "lifetime_high":         [ 80,           1680,         12000,        2,            2           ],
})

# ── DNBI diagnosis mix ────────────────────────────────────────────────────────
dnbi_mix_df = pd.DataFrame({
    "triage":                ["T1_DNBI",    "T2_DNBI",    "T3_DNBI"  ],
    "frequency":             [ 0.15,         0.20,         0.65       ],
    "lifetime_distribution": ["triangular",  "triangular", "triangular"],
    "lifetime_low":          [ 40,           1200,         10000      ],
    "lifetime_mode":         [ 60,           1440,         11000      ],
    "lifetime_high":         [ 80,           1680,         12000      ],
})

# ── Prescriptions ─────────────────────────────────────────────────────────────
prescriptions_df = pd.DataFrame({
    "label": [
        "T1_BCAS", "T1_BCAS",
        "T2_BCAS", "T2_BCAS",
        "T3_BCAS", "T3_BCAS",
        "T1_DNBI", "T1_DNBI",
        "T2_DNBI", "T2_DNBI",
        "T3_DNBI", "T3_DNBI",
        "KIA", "MIA",
    ],
    "frequency":   [0.95, 0.05, 0.80, 0.20, 0.80, 0.20, 0.95, 0.05, 0.80, 0.20, 0.20, 0.80, 1.00, 1.00],
    "prescription": [
        ["DCR", "DCS"], ["DCR"],
        ["DCR", "DCS"], ["DCR"],
        ["PEC", "DCS"], ["PEC"],
        ["DCR", "DCS"], ["DCR"],
        ["DCR", "DCS"], ["DCR"],
        ["PEC", "DCS"], ["PEC"],
        [], [],
    ],
})

# ── Streams: derived from force structure loaded in cell 1 ────────────────────
# PAR_TOTAL is the sum of all involved battalions
PAR_TOTAL = int(involved["par"].sum())

# BCAS: spawn at the CCPs of each involved battalion
STREAMS = [
    dict(name="BCAS", facility_ids=inv_spawn, rate=0.023, mix_df=bcas_mix_df),
]

In [209]:
# ── Expected casualties breakdown ─────────────────────────────────────────────
# One row per battalion, with per-CCP and per-wave per-triage breakdown

# Build triage frequencies (excluding KIA/MIA — simulation input only)
sim_triage_rows = bcas_mix_df[~bcas_mix_df["triage"].isin(["KIA","MIA"])].copy()
sim_triage_rows["freq_norm"] = sim_triage_rows["frequency"] / sim_triage_rows["frequency"].sum()

kia_mia_rate   = bcas_mix_df[bcas_mix_df["triage"].isin(["KIA","MIA"])]["frequency"].sum()
sim_casualty_rate = STREAMS[0]["rate"] * (1 - kia_mia_rate)   # survivors only

rows = []
for _, bn in fs_df.iterrows():
    if not bn["involved"]:
        continue
    stream_rate = STREAMS[0]["rate"]
    ccps        = bn["spawning_points"]
    n_ccps      = len(ccps)
    n_waves     = N_DAYS * WAVES_PER_DAY

    exp_total   = bn["par"] * stream_rate * N_DAYS           # total expected (incl KIA/MIA)
    exp_sim     = bn["par"] * sim_casualty_rate * N_DAYS     # simulation input (excl KIA/MIA)
    exp_per_ccp = exp_sim / n_ccps if n_ccps else 0
    exp_per_wave_per_ccp = exp_per_ccp / n_waves if n_waves else 0

    triage_per_wave = {
        row["triage"]: exp_per_wave_per_ccp * row["freq_norm"]
        for _, row in sim_triage_rows.iterrows()
    }

    row = {
        "battalion":          bn["battalion"],
        "brigade":            bn["brigade"],
        "par":                int(bn["par"]),
        "n_ccps":             n_ccps,
        "exp_total":          round(exp_total, 1),
        "exp_sim_input":      round(exp_sim, 1),
        "exp_per_ccp":        round(exp_per_ccp, 2),
        "exp_per_wave/ccp":   round(exp_per_wave_per_ccp, 2),
    }
    for t, v in triage_per_wave.items():
        row[f"wave/{t}"] = round(v, 3)
    rows.append(row)

exp_df = pd.DataFrame(rows)
print("Expected casualties per battalion")
print(f"  Rate: {STREAMS[0]['rate']*100:.3f}% BCAS/soldier/day  |  "
      f"N_DAYS={N_DAYS}  WAVES/DAY={WAVES_PER_DAY}  WAVE_DUR={WAVE_DURATION_MIN} min")
print()
print(exp_df.to_string(index=False))

Expected casualties per battalion
  Rate: 2.300% BCAS/soldier/day  |  N_DAYS=1  WAVES/DAY=1  WAVE_DUR=30 min

battalion brigade  par  n_ccps  exp_total  exp_sim_input  exp_per_ccp  exp_per_wave/ccp  wave/T1_BCAS  wave/T2_BCAS  wave/T3_BCAS
      BN3    BDE2 3133       3       72.1           50.4        16.81             16.81         2.522         3.363        10.929


In [210]:
# ============================================================
# DATA MODEL
# ============================================================

@dataclass
class Wave:
    wave_id: str
    t_start: float          # minutes from sim start
    t_end: float            # minutes from sim start
    facility_id: str
    n_casualties: int = 0
    df: Optional[pd.DataFrame] = None  # at least: t_create, spawn_facility_id


WaveDict = Dict[str, List[Wave]]  # facility_id -> list[Wave]

In [211]:
# ============================================================
# HELPERS
# ============================================================

def _normalize_probs(p: np.ndarray) -> np.ndarray:
    s = float(p.sum())
    if s <= 0:
        raise ValueError("Probabilities must sum to > 0")
    return p / s

In [212]:
# ============================================================
# LIFETIME FACTORY BUILDER
# ============================================================

def _lifetime_df_to_factories(diagnosis_mix_df: pd.DataFrame) -> Dict[str, Callable[[], object]]:
    """Convert diagnosis_mix dataframe to lifetime factory functions."""
    factories = {}

    for _, row in diagnosis_mix_df.iterrows():
        triage = row["triage"]
        dist_type = row["lifetime_distribution"].lower()

        if dist_type == "uniform":
            low, high = row["lifetime_low"], row["lifetime_high"]
            factories[triage] = lambda l=low, h=high: sim.Uniform(l, h)
        elif dist_type == "triangular":
            low = row["lifetime_low"]
            high = row["lifetime_high"]
            mode = row.get("lifetime_mode", (low + high) / 2)
            if pd.isna(mode):
                mode = (low + high) / 2
            factories[triage] = lambda l=low, m=mode, h=high: sim.Triangular(l, h, m)
        elif dist_type == "exponential":
            mean = row["lifetime_mean"]
            factories[triage] = lambda m=mean: sim.Exponential(m)
        elif dist_type == "normal":
            mean, std = row["lifetime_mean"], row["lifetime_std"]
            factories[triage] = lambda m=mean, s=std: sim.Normal(m, s)
        else:
            raise ValueError(f"Unknown distribution type: {dist_type}")

    return factories

In [213]:
# ============================================================
# STEP A) WAVE PLACER
#   - creates wave_dict with Wave objects (no casualties yet)
#   - random placement per day per facility
#   - NO overlap by construction using random slot selection
# ============================================================

def place_waves_randomly_per_day(
    facility_ids: List[str],
    n_days: int,
    waves_per_day: int,
    wave_duration_min: int,
    seed: int = 123,
) -> WaveDict:
    """
    Places 'waves_per_day' non-overlapping waves of duration 'wave_duration_min'
    randomly within each day for each facility.

    Slot method:
      day split into S = floor(1440 / wave_duration_min) slots.
      For each (facility, day) sample 'waves_per_day' distinct slots.
    """
    rng = random.Random(seed)
    minutes_per_day = 24 * 60

    if wave_duration_min <= 0:
        raise ValueError("wave_duration_min must be > 0")
    if n_days <= 0:
        raise ValueError("n_days must be > 0")
    if waves_per_day <= 0:
        raise ValueError("waves_per_day must be > 0")
    if not facility_ids:
        raise ValueError("facility_ids must be a non-empty list")

    slots_per_day = minutes_per_day // wave_duration_min
    if waves_per_day > slots_per_day:
        raise ValueError(
            f"waves_per_day={waves_per_day} too large for wave_duration_min={wave_duration_min}. "
            f"Max is {slots_per_day}."
        )

    wave_dict: WaveDict = {fid: [] for fid in facility_ids}
    wave_counter = 0
    slot_indices = list(range(slots_per_day))

    for day in range(n_days):
        day_start = day * minutes_per_day
        for fid in facility_ids:
            chosen_slots = rng.sample(slot_indices, k=waves_per_day)
            chosen_slots.sort()
            for s in chosen_slots:
                t_start = day_start + s * wave_duration_min
                t_end = t_start + wave_duration_min
                wave_id = f"w_{wave_counter:06d}"
                wave_counter += 1
                wave_dict[fid].append(Wave(
                    wave_id=wave_id,
                    t_start=float(t_start),
                    t_end=float(t_end),
                    facility_id=fid,
                ))

    return wave_dict

In [214]:

# ============================================================
# STEP B) WAVE CASUALTY ALLOCATOR
#
# For each day:
#   N_day_total ~ Poisson(par_total * rate_per_day)
# Distribute across facilities (weighted multinomial),
# then across that facility's waves (random multinomial).
# ============================================================

def allocate_casualties_to_waves(
    wave_dict: WaveDict,
    par_total: int,
    rate_per_day: float,
    facility_weights: Optional[Dict[str, float]] = None,
    seed: int = 123,
) -> None:
    """
    Mutates wave_dict in-place by setting wave.n_casualties.

    - par_total:        global PAR (e.g., 18800)
    - rate_per_day:     casualties per soldier per day
    - facility_weights: optional weights per facility_id; None = equal weights
    """
    if par_total < 0:
        raise ValueError("par_total must be >= 0")
    if rate_per_day < 0:
        raise ValueError("rate_per_day must be >= 0")

    rng = np.random.default_rng(seed)
    minutes_per_day = 24 * 60
    facility_ids = list(wave_dict.keys())
    if not facility_ids:
        return

    if facility_weights is None:
        weights = np.ones(len(facility_ids), dtype=float)
    else:
        weights = np.array(
            [float(facility_weights.get(fid, 0.0)) for fid in facility_ids], dtype=float
        )
        if (weights < 0).any():
            raise ValueError("facility_weights must be non-negative")
        if float(weights.sum()) <= 0:
            raise ValueError("facility_weights sum must be > 0 (or pass None for equal split)")

    weights = _normalize_probs(weights)

    all_days = set()
    for fid in facility_ids:
        for w in wave_dict[fid]:
            all_days.add(int(w.t_start // minutes_per_day))
    if not all_days:
        return
    n_days = max(all_days) + 1

    for day_idx in range(n_days):
        lam_day = par_total * rate_per_day
        n_day_total = int(rng.poisson(lam=lam_day))

        if n_day_total <= 0:
            for fid in facility_ids:
                for w in wave_dict[fid]:
                    if int(w.t_start // minutes_per_day) == day_idx:
                        w.n_casualties = 0
            continue

        # Split across facilities
        n_by_facility = rng.multinomial(n_day_total, weights)

        for fid, n_fac_day in zip(facility_ids, n_by_facility):
            day_waves = [
                w for w in wave_dict[fid]
                if int(w.t_start // minutes_per_day) == day_idx
            ]
            if not day_waves:
                continue

            day_waves.sort(key=lambda ww: ww.t_start)
            k = len(day_waves)

            # Random split across waves (multinomial, equal weights)
            wave_weights = np.ones(k, dtype=float) / k
            counts = rng.multinomial(int(n_fac_day), wave_weights)
            for w, c in zip(day_waves, counts):
                w.n_casualties = int(c)


In [215]:
# ============================================================
# STEP C) WAVE FILLER
#   - creates wave.df with ONLY: t_create, spawn_facility_id
#   - inside each wave: conditional on n, times ~ Uniform(t_start, t_end)
# ============================================================

def fill_wave_dataframes(wave_dict: WaveDict, seed: int = 123) -> None:
    rng = np.random.default_rng(seed)

    for fid, waves in wave_dict.items():
        for w in waves:
            n = int(w.n_casualties)
            if n <= 0:
                w.df = pd.DataFrame(columns=["t_create", "spawn_facility_id"])
                continue
            t = rng.uniform(low=w.t_start, high=w.t_end, size=n)
            w.df = (
                pd.DataFrame({"t_create": t.astype(float), "spawn_facility_id": [fid] * n})
                .sort_values("t_create")
                .reset_index(drop=True)
            )

In [216]:
# ============================================================
# STEP D) CASUALTY POPULATOR
#   Adds: triage, prescription, lifetime_dist
# ============================================================

def populate_patient_attributes(
    wave_dict: WaveDict,
    diagnosis_mix_df: pd.DataFrame,
    rx_df: pd.DataFrame,
    lifetime_factory_by_triage: Optional[Dict[str, Callable[[], object]]] = None,
    seed: int = 123,
) -> None:
    rng = np.random.default_rng(seed)

    diag_labels = diagnosis_mix_df["triage"].astype(str).to_numpy()
    diag_p = _normalize_probs(diagnosis_mix_df["frequency"].astype(float).to_numpy())

    if lifetime_factory_by_triage is None:
        lifetime_factory_by_triage = {}
    default_lifetime_factory = lambda: sim.Uniform(60, 120)

    rx_df = rx_df.copy()
    rx_df["label"] = rx_df["label"].astype(str)

    rx_by_label: Dict[str, Tuple[np.ndarray, List[List[str]]]] = {}
    for label, g in rx_df.groupby("label"):
        presc_list = [list(x) for x in g["prescription"].tolist()]
        p = _normalize_probs(g["frequency"].astype(float).to_numpy())
        rx_by_label[label] = (p, presc_list)

    global_presc_list = [list(x) for x in rx_df["prescription"].tolist()]
    global_p = _normalize_probs(rx_df["frequency"].astype(float).to_numpy())

    for _, waves in wave_dict.items():
        for w in waves:
            if w.df is None or len(w.df) == 0:
                if w.df is None:
                    w.df = pd.DataFrame(columns=["t_create", "spawn_facility_id"])
                w.df["triage"] = pd.Series(dtype=str)
                w.df["prescription"] = pd.Series(dtype=object)
                w.df["lifetime_dist"] = pd.Series(dtype=object)
                continue

            n = len(w.df)
            triage = rng.choice(diag_labels, size=n, p=diag_p).astype(str)

            prescriptions: List[List[str]] = []
            lifetime_dists: List[object] = []

            for tr in triage:
                if tr in rx_by_label:
                    p, presc_list = rx_by_label[tr]
                    idx = int(rng.choice(np.arange(len(presc_list)), p=p))
                    prescriptions.append(list(presc_list[idx]))
                else:
                    idx = int(rng.choice(np.arange(len(global_presc_list)), p=global_p))
                    prescriptions.append(list(global_presc_list[idx]))

                lf_factory = lifetime_factory_by_triage.get(tr, default_lifetime_factory)
                lifetime_dists.append(lf_factory())

            w.df = w.df.copy()
            w.df["triage"] = triage
            w.df["prescription"] = prescriptions
            w.df["lifetime_dist"] = lifetime_dists

In [217]:

# ============================================================
# STEP E) MERGE WAVES -> FINAL casualties_df
#   Accepts one or more WaveDicts (e.g. BCAS + DNBI separately)
# ============================================================

def merge_waves_to_casualties_df(*wave_dicts: WaveDict) -> pd.DataFrame:
    frames = []
    for wave_dict in wave_dicts:
        for waves in wave_dict.values():
            for w in waves:
                if w.df is not None and len(w.df) > 0:
                    frames.append(w.df)

    if not frames:
        return pd.DataFrame(
            columns=["t_create", "spawn_facility_id", "triage", "prescription", "lifetime_dist"]
        )

    out = pd.concat(frames, ignore_index=True)
    return out.sort_values("t_create").reset_index(drop=True)


In [218]:
# ============================================================
# SAVE HELPER
# ============================================================

def save_casualties_df(
    casualties_df: pd.DataFrame,
    output_path: Union[str, Path],
) -> Path:
    """Save casualties dataframe to XLSX, extracting lifetime bounds."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    save_df = casualties_df.copy()

    def get_dist_type(d):
        cls_name = type(d).__name__.lower()
        if "triangular"   in cls_name: return "triangular"
        elif "uniform"    in cls_name: return "uniform"
        elif "exponential"in cls_name: return "exponential"
        elif "normal"     in cls_name: return "normal"
        return "unknown"

    def get_low(d):
        # salabim uses _low; fall back to common alternatives
        for attr in ["_low", "_lowerbound", "low", "a", "min"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    def get_high(d):
        # salabim uses _high; fall back to common alternatives
        for attr in ["_high", "_upperbound", "high", "b", "max"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    def get_mode(d):
        for attr in ["_mode", "mode", "c"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    save_df["lifetime_dist_type"] = save_df["lifetime_dist"].apply(get_dist_type)
    save_df["lifetime_low"]       = save_df["lifetime_dist"].apply(get_low)
    save_df["lifetime_high"]      = save_df["lifetime_dist"].apply(get_high)
    save_df["lifetime_mode"]      = save_df["lifetime_dist"].apply(get_mode)

    save_cols = [c for c in save_df.columns if c != "lifetime_dist"]
    save_df[save_cols].to_excel(output_path, index=False)
    print(f"Saved {len(save_df)} rows to: {output_path}")
    return output_path

In [219]:

# ============================================================
# ORCHESTRATOR
#   Runs the full pipeline for BCAS (CCPs) and DNBI (Role 1s)
#   separately, then merges into one casualties_df.
# ============================================================

def generate_casualties_df_from_waves(
    ccp_ids: List[str],
    role1_ids: List[str],
    par_total: int,
    n_days: int,
    waves_per_day: int,
    wave_duration_min: int,
    rate_bcas: float,
    rate_dnbi: float,
    bcas_mix_df: pd.DataFrame,
    dnbi_mix_df: pd.DataFrame,
    rx_df: pd.DataFrame,
    facility_weights_ccp: Optional[Dict[str, float]] = None,
    facility_weights_r1:  Optional[Dict[str, float]] = None,
    lifetime_factory_by_triage: Optional[Dict[str, Callable[[], object]]] = None,
    seed: int = 123,
) -> Tuple[pd.DataFrame, WaveDict, WaveDict]:
    """
    Returns (casualties_df, wave_dict_bcas, wave_dict_dnbi).

    casualties_df contains ALL generated entities including KIA and MIA.
    Filter with  simulation_input_df = casualties_df[~casualties_df['triage'].isin(['KIA','MIA'])]
    before passing to the simulator.
    """

    # ── BCAS: battle casualties spawn at CCPs ────────────────────────────────
    wave_dict_bcas = place_waves_randomly_per_day(
        facility_ids=ccp_ids,
        n_days=n_days,
        waves_per_day=waves_per_day,
        wave_duration_min=wave_duration_min,
        seed=seed,
    )
    allocate_casualties_to_waves(
        wave_dict=wave_dict_bcas,
        par_total=par_total,
        rate_per_day=rate_bcas,
        facility_weights=facility_weights_ccp,
        seed=seed + 1,
    )
    fill_wave_dataframes(wave_dict=wave_dict_bcas, seed=seed + 2)
    populate_patient_attributes(
        wave_dict=wave_dict_bcas,
        diagnosis_mix_df=bcas_mix_df,
        rx_df=rx_df,
        lifetime_factory_by_triage=lifetime_factory_by_triage,
        seed=seed + 3,
    )

    # ── DNBI: disease/non-battle casualties spawn at Role 1 MTFs ────────────
    wave_dict_dnbi = place_waves_randomly_per_day(
        facility_ids=role1_ids,
        n_days=n_days,
        waves_per_day=waves_per_day,
        wave_duration_min=wave_duration_min,
        seed=seed + 10,
    )
    allocate_casualties_to_waves(
        wave_dict=wave_dict_dnbi,
        par_total=par_total,
        rate_per_day=rate_dnbi,
        facility_weights=facility_weights_r1,
        seed=seed + 11,
    )
    fill_wave_dataframes(wave_dict=wave_dict_dnbi, seed=seed + 12)
    populate_patient_attributes(
        wave_dict=wave_dict_dnbi,
        diagnosis_mix_df=dnbi_mix_df,
        rx_df=rx_df,
        lifetime_factory_by_triage=lifetime_factory_by_triage,
        seed=seed + 13,
    )

    # ── Merge ────────────────────────────────────────────────────────────────
    casualties_df = merge_waves_to_casualties_df(wave_dict_bcas, wave_dict_dnbi)
    return casualties_df, wave_dict_bcas, wave_dict_dnbi


In [220]:
# ============================================================
# RUN
# ============================================================

OUTPUT_PATH = f"data/single_run/input/casualty_dataframes/{OUTPUT_NAME}.xlsx"

all_factories = {}
for stream in STREAMS:
    all_factories.update(_lifetime_df_to_factories(stream["mix_df"]))

wave_dicts = []
for i, stream in enumerate(STREAMS):
    base_seed = SEED + i * 10
    wdict = place_waves_randomly_per_day(
        facility_ids=stream["facility_ids"],
        n_days=N_DAYS,
        waves_per_day=WAVES_PER_DAY,
        wave_duration_min=WAVE_DURATION_MIN,
        seed=base_seed,
    )
    allocate_casualties_to_waves(wdict, PAR_TOTAL, stream["rate"], seed=base_seed + 1)
    fill_wave_dataframes(wdict, seed=base_seed + 2)
    populate_patient_attributes(wdict, stream["mix_df"], prescriptions_df, all_factories, seed=base_seed + 3)
    wave_dicts.append(wdict)
    print(f"Stream '{stream['name']}': {sum(len(w) for w in wdict.values())} waves across {len(wdict)} facilities")

casualties_df = merge_waves_to_casualties_df(*wave_dicts)

simulation_input_df = casualties_df[
    ~casualties_df["triage"].isin(["KIA", "MIA"])
].copy().reset_index(drop=True)

if SAVE_OUTPUT:
    save_casualties_df(simulation_input_df, OUTPUT_PATH)

print("\n=== Triage breakdown (all generated) ===")
print(casualties_df["triage"].value_counts().sort_index().to_string())
print(f"\nTotal generated:  {len(casualties_df)}")
print(f"  KIA:            {(casualties_df['triage'] == 'KIA').sum()}")
print(f"  MIA:            {(casualties_df['triage'] == 'MIA').sum()}")
print(f"\nSimulation input: {len(simulation_input_df)}")
print(f"\nSpawn locations:")
print(simulation_input_df["spawn_facility_id"].value_counts().sort_index().to_string())

Stream 'BCAS': 3 waves across 3 facilities
Saved 45 rows to: data\single_run\input\casualty_dataframes\casualties_custom_test.xlsx

=== Triage breakdown (all generated) ===
triage
KIA        15
MIA         3
T1_BCAS     7
T2_BCAS    10
T3_BCAS    28

Total generated:  63
  KIA:            15
  MIA:            3

Simulation input: 45

Spawn locations:
spawn_facility_id
ccp_7    12
ccp_8    13
ccp_9    20


In [221]:
# ── Sanity check ──────────────────────────────────────────────────────────────
import math

ok   = "  ✓"
fail = "  ✗"

def check(label, passed, detail=""):
    print(f"{ok if passed else fail}  {label}" + (f"  →  {detail}" if detail else ""))

print("=" * 60)
print("  SANITY CHECK")
print("=" * 60)

n_total       = len(casualties_df)
total_expected = inv_par * STREAMS[0]["rate"] * N_DAYS
tol_total      = 0.15   # 15% tolerance on total count

# 1. Total count plausible
dev = abs(n_total - total_expected) / max(total_expected, 1)
check(
    f"Total casualties: {n_total}  (expected ≈ {total_expected:.0f},  dev={dev*100:.1f}%)",
    dev <= tol_total
)

# 2. Triage mix — tolerance scales with sample size (2 std devs of binomial)
print()
print("  Triage mix (actual vs expected, tolerance = 2σ):")
for _, mrow in bcas_mix_df.iterrows():
    t        = mrow["triage"]
    p_exp    = mrow["frequency"]
    n_act    = (casualties_df["triage"] == t).sum()
    p_act    = n_act / n_total if n_total else 0
    sigma    = math.sqrt(p_exp * (1 - p_exp) / max(n_total, 1))
    tol      = 2 * sigma
    passed   = abs(p_act - p_exp) <= tol
    check(
        f"  {t:<12} actual={p_act*100:5.1f}%  expected={p_exp*100:5.1f}%"
        f"  ±{tol*100:.1f}%  (n={n_act})",
        passed
    )

# 3. All involved spawning points present
print()
actual_locs = set(casualties_df["spawn_facility_id"].unique())
missing = set(inv_spawn) - actual_locs
extra   = actual_locs - set(inv_spawn)
check(f"All {len(inv_spawn)} spawning points populated", not missing,
      f"missing: {missing}" if missing else "")
check(f"No unexpected spawn locations", not extra,
      f"extra: {extra}" if extra else "")

# 4. Timing within simulation bounds
sim_till = N_DAYS * 1440
t_min = simulation_input_df["t_create"].min()
t_max = simulation_input_df["t_create"].max()
check(f"All t_create within [0, {sim_till}] min  (actual [{t_min:.0f}, {t_max:.0f}])",
      t_min >= 0 and t_max <= sim_till)

# 5. No NaN in lifetime parameters
for col in ["lifetime_low", "lifetime_high", "lifetime_mode"]:
    n_nan = simulation_input_df[col].isna().sum() if col in simulation_input_df.columns else 0
    check(f"No NaN in {col}", n_nan == 0, f"{n_nan} NaNs" if n_nan else "")

# 6. No empty prescriptions
empty_rx = simulation_input_df["prescription"].apply(
    lambda x: len(eval(x) if isinstance(x, str) else x) == 0
).sum()
check(f"No empty prescriptions", empty_rx == 0, f"{empty_rx} empty" if empty_rx else "")

print()
print("=" * 60)

  SANITY CHECK
  ✓  Total casualties: 63  (expected ≈ 72,  dev=12.6%)

  Triage mix (actual vs expected, tolerance = 2σ):
  ✓    T1_BCAS      actual= 11.1%  expected= 10.5%  ±7.7%  (n=7)
  ✓    T2_BCAS      actual= 15.9%  expected= 14.0%  ±8.7%  (n=10)
  ✓    T3_BCAS      actual= 44.4%  expected= 45.5%  ±12.5%  (n=28)
  ✓    KIA          actual= 23.8%  expected= 20.0%  ±10.1%  (n=15)
  ✓    MIA          actual=  4.8%  expected= 10.0%  ±7.6%  (n=3)

  ✓  All 3 spawning points populated
  ✓  No unexpected spawn locations
  ✓  All t_create within [0, 1440] min  (actual [842, 1079])
  ✓  No NaN in lifetime_low
  ✓  No NaN in lifetime_high
  ✓  No NaN in lifetime_mode
  ✓  No empty prescriptions



In [222]:
import plotly.graph_objects as go

all_waves: dict = {}
fac_stream: dict = {}
STREAM_COLORS = [
    "rgba(180, 40,  30,  0.95)",
    "rgba( 30,100, 190, 0.95)",
    "rgba( 30,160,  80, 0.95)",
    "rgba(200,150,   0, 0.95)",
]

for i, (stream, wdict) in enumerate(zip(STREAMS, wave_dicts)):
    color = STREAM_COLORS[i % len(STREAM_COLORS)]
    for fac_id, waves in wdict.items():
        all_waves.setdefault(fac_id, []).extend(waves)
        fac_stream[fac_id] = (stream["name"], color)

locations = sorted(all_waves.keys())
loc_index = {loc: i for i, loc in enumerate(locations)}
max_time  = max(w.t_end for waves in all_waves.values() for w in waves)

fig = go.Figure()

for fac_id, waves in all_waves.items():
    y = loc_index[fac_id]
    _, color = fac_stream[fac_id]
    for w in waves:
        fig.add_shape(type="rect", x0=w.t_start, x1=w.t_end,
                      y0=y - 0.4, y1=y + 0.4, fillcolor=color,
                      line=dict(width=0.5, color="black"))

hover_x, hover_y, hover_text = [], [], []
for fac_id, waves in all_waves.items():
    y = loc_index[fac_id]
    for w in waves:
        hover_x.append((w.t_start + w.t_end) / 2)
        hover_y.append(y)
        counts = (w.df["triage"].value_counts().sort_index().to_dict()
                  if w.df is not None and "triage" in w.df.columns and len(w.df) > 0 else {})
        triage_lines = "<br>".join(f"&nbsp;&nbsp;{t}: {n}" for t, n in sorted(counts.items()))
        hover_text.append(f"<b>{fac_id}</b><br>t: {w.t_start:.0f}–{w.t_end:.0f} min<br>"
                          f"total: {sum(counts.values())}<br>{triage_lines}")

fig.add_trace(go.Scatter(x=hover_x, y=hover_y, mode="markers",
    marker=dict(size=18, opacity=0), hovertemplate="%{text}<extra></extra>",
    text=hover_text, showlegend=False))

for d in range(1, int(max_time // 1440) + 2):
    fig.add_vline(x=d * 1440, line=dict(color="grey", dash="dash", width=1))

seen = set()
for fac_id in locations:
    stream, color = fac_stream[fac_id]
    if stream not in seen:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
            marker=dict(size=12, color=color, symbol="square"),
            name=stream, showlegend=True))
        seen.add(stream)

fig.update_layout(
    title="Casualty Generation Waves",
    xaxis=dict(title=dict(text="Simulation time (min)", font=dict(color="black")),
               tickfont=dict(color="black"), range=[0, max_time]),
    yaxis=dict(title=dict(text="Spawn location", font=dict(color="black")),
               tickfont=dict(color="black"),
               tickvals=list(range(len(locations))), ticktext=locations,
               range=[-0.5, len(locations) - 0.5]),
    height=max(400, len(locations) * 22 + 120),
    hovermode="closest", plot_bgcolor="white", paper_bgcolor="white",
    font=dict(color="black"),
)
fig.show()

In [223]:
# ============================================================
# VISUALIZATION — Casualties per wave slot
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── SELECT ────────────────────────────────────────────────────────────────────
PLOT_FACILITY = "ccp_10"   # any facility_id from any stream
PLOT_DAY      = 0        # 0-indexed
# ─────────────────────────────────────────────────────────────────────────────

all_wave_dicts = {}
for wdict in wave_dicts:
    all_wave_dicts.update(wdict)

if PLOT_FACILITY not in all_wave_dicts:
    raise ValueError(f"'{PLOT_FACILITY}' not found. Available: {list(all_wave_dicts.keys())}")

minutes_per_day = 24 * 60
slots_per_day   = minutes_per_day // WAVE_DURATION_MIN
day_start       = PLOT_DAY * minutes_per_day

slot_counts = np.zeros(slots_per_day, dtype=int)
for w in all_wave_dicts[PLOT_FACILITY]:
    if int(w.t_start // minutes_per_day) == PLOT_DAY:
        slot_idx = int((w.t_start - day_start) // WAVE_DURATION_MIN)
        slot_counts[slot_idx] += w.n_casualties

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["steelblue" if c > 0 else "lightgrey" for c in slot_counts]
ax.bar(range(slots_per_day), slot_counts, color=colors, edgecolor="white", linewidth=0.5)
for i, c in enumerate(slot_counts):
    if c > 0:
        ax.text(i, c + 0.3, str(c), ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xlabel(f"Slot index  (each slot = {WAVE_DURATION_MIN} min)")
ax.set_ylabel("Casualties")
ax.set_title(f"Casualties per {WAVE_DURATION_MIN}-min slot — facility: {PLOT_FACILITY}, day {PLOT_DAY}")
ax.set_xlim(-0.5, slots_per_day - 0.5)
ax.yaxis.get_major_locator().set_params(integer=True)

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
tick_pos = list(range(0, slots_per_day, max(1, slots_per_day // 12)))
ax2.set_xticks(tick_pos)
ax2.set_xticklabels([f"{(t * WAVE_DURATION_MIN) // 60:02d}:00" for t in tick_pos], fontsize=8)
ax2.set_xlabel("Time of day")

ax.legend(handles=[
    mpatches.Patch(color="steelblue", label="Wave slots"),
    mpatches.Patch(color="lightgrey", label="Empty slots"),
], loc="upper right")
plt.tight_layout()
plt.show()
print(f"Total: {slot_counts.sum()} | Active slots: {(slot_counts > 0).sum()} / {slots_per_day}")

ValueError: 'ccp_10' not found. Available: ['ccp_7', 'ccp_8', 'ccp_9']